In [ ]:
from pathlib import Path
from collections import defaultdict
import shelve
import shutil

from PIL import Image
import numpy as np
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import umap

from is_image_document_ai import infer_folder, infer_single_image

In [ ]:
MEDIA_PATH = Path.home() / ".PhotoChart" / "database" / "media" / "photographs"

In [ ]:
paths = []
for root, dirs, files in MEDIA_PATH.walk():
    paths.extend([root / f for f in files])

In [ ]:
infer_single_image(paths[1])

In [ ]:
def image_histogram(img, bins=np.linspace(0, 255.0, 18)):
    img = np.asarray(img)
    return np.array([
        np.histogram(img[..., i], bins=bins)[0]
        for i in range(img.shape[-1])
    ])

def get_media_rgb_histograms():
    hists = []
    shapes = defaultdict(list)
    paths1 = []
    for path in tqdm(paths[:6000]):
        img = Image.open(path)
        shape = np.asarray(img).shape
        if len(shape) != 3:
            # Do not consider 2D images (only RGB)
            continue
        if shape[-1] != 3:
            # Do not consider 4th channel (no RGBA)
            continue
        shapes[shape].append(path)
        hists.append(image_histogram(img))
        paths1.append(path)
    hists = np.array(hists)
    hists = hists / hists.sum(axis=-1, keepdims=True)
    return hists, paths1

In [ ]:
with shelve.open("ShelveObjects") as db:
    if "hists_paths" in db:
        hists, paths = db["hists_paths"]
    else:
        hists, paths = get_media_histograms()
        db["hists_paths"] = hists, paths

In [ ]:
n_plot = 40
hists_plot = hists[:n_plot]
N, channels, bins = hists_plot.shape

fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection='3d')

colors = ['red', 'green', 'blue']

# Bar dimensions
dx = 0.8
dy = 0.25

x = np.arange(bins)

for i in range(N):
    for c in range(channels):
        y = i + c * dy

        ax.bar3d(
            x,
            np.full(bins, y),
            np.zeros(bins),
            dx,
            dy,
            hists_plot[i, c, :],
            color=colors[c],
            alpha=0.8,
            shade=True
        )

ax.set_xlabel('Histogram bin')
ax.set_ylabel('Image index')
ax.set_zlabel('Pixel count')

ax.set_title('RGB Histograms Across Images')

# Show image indices as depth-axis ticks
ax.set_yticks(np.arange(N) + dy)
ax.set_yticklabels(np.arange(N))

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

# hists shape: (n_images, 3, n_bins)

# Normalize each color-channel histogram
hists_norm = hists / np.maximum(hists.sum(axis=2, keepdims=True), 1e-12)

# Flatten into feature vectors: (n_images, 3 * n_bins)
X = hists_norm.reshape(len(hists_norm), -1)

# Choose the number of clusters
n_clusters = 8

# Cluster the images
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
labels = kmeans.fit_predict(X)

In [ ]:
OUTPUT = Path.cwd() / "output" / f"KMeans_{n_clusters}"
OUTPUT.mkdir(parents=True, exist_ok=True)
for original, label in tqdm(zip(paths, labels), total=len(paths)):
    label_dir = OUTPUT / str(label)
    label_dir.mkdir(parents=False, exist_ok=True)
    shutil.copy2(original, label_dir)

In [ ]:
# Project features into 3D
X_pca = PCA(n_components=3).fit_transform(X)

# Plot
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")

scatter = ax.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    X_pca[:, 2],
    c=labels,
    cmap="tab10",
    s=50,
    alpha=0.8
)

ax.set_title(f"PCA projection of image histograms ({n_clusters} clusters)")
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")

fig.colorbar(scatter, ax=ax, label="Cluster", pad=0.1)
plt.tight_layout()
plt.show()

In [ ]:
import umap

# Project features into 3D
reducer = umap.UMAP(
    n_components=3,
    n_neighbors=15,
    min_dist=0.1,
    metric="euclidean",
    random_state=42
)

X_umap = reducer.fit_transform(X)

# Plot
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")

scatter = ax.scatter(
    X_umap[:, 0],
    X_umap[:, 1],
    X_umap[:, 2],
    c=labels,
    cmap="tab10",
    s=50,
    alpha=0.8
)

ax.set_title(f"UMAP projection of image histograms ({n_clusters} clusters)")
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
ax.set_zlabel("UMAP 3")

fig.colorbar(scatter, ax=ax, label="Cluster", pad=0.1)
plt.tight_layout()
plt.show()

In [ ]:
OUTPUT = Path.cwd() / "output" / f"is_image_document_ai"
OUTPUT.mkdir(parents=True, exist_ok=True)
for path in tqdm(paths):
    label = infer_single_image(path)
    label_dir = OUTPUT / str(label)
    label_dir.mkdir(parents=False, exist_ok=True)
    shutil.copy2(path, label_dir)